In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

c:\Users\Admin\anaconda3\envs\t5-summarizer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_data = pd.read_csv("../data/samsum-train.csv")
val_data = pd.read_csv("../data/samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.sample(10)

,id,dialogue,summary
14693,13812564,Maria: Did you note down the lecture?\r\nJesus...,Professor was speaking too fast to take notes....
13921,13717356,Lance: I caved in... :P\r\nLance: i ordered th...,Lance bought a Philips TV. Ricky and Neil are ...
9066,13730991,"Fiona: Hello Pat, you coming to book group ton...",Fiona and Pat will see each other tonight at S...
5494,13716515,"Josh: <file_photo>\r\nJosh: hi guys, do you kn...",Josh met the man involved in the whiskey incid...
10354,13864809,Lola: <file_other>\nLola: check it!\nLaura: hm...,Jeff doesn't like Lonely Planet. Lola is going...
5391,13728692-1,"Jorge: Hurry up, man\r\nDusan: I'm trying, I'm...",Dusan is on his way to the show he is going to...
7610,13821688,"Dan: Hi guys\r\nLautaro: 👋\r\nPablo: Hey Dan, ...",Lautaro is going to have a few drinks in Paler...
12520,13865222,Felix: Do you know this guy:\nFelix: <file_oth...,Felix sends a file with an indietronica artist...
11914,13680813,Alan: Going to be late for the Tennis club!\r\...,Alan is going to be 15 minutes late for the Te...
13775,13682105,Jack: Look at the woman next to me :P\r\nJill:...,The woman next to Jack looks like she had 555 ...


In [6]:
print(train_data.shape)
print(val_data.shape)

(14732, 3)
(818, 3)


In [7]:
# random sampling

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [8]:
print(train_data.shape)
print(val_data.shape)

(4000, 3)
(500, 3)


# Data Preprocessing

In [9]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)  # lines
    text = re.sub(r"\s+", " ", text)   # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    text = text.strip().lower()

    return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## Tokenize

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [13]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as lables

    return inputs

In [14]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [15]:
train_dataset[0]

# input_ids - dialogue => token ids
# attention mask
# labels - target => summary

# 1 = end of sequence (EOS)
# 0 = padding

# for attention mask
# 1 - valid token ids
# 0 - padding

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
print(len(train_dataset[0]["input_ids"]))
print(len(train_dataset[0]["labels"]))

512
150


In [17]:
print(type(train_dataset))
print(type(val_dataset))

<class 'list'>
<class 'list'>


# Working with our model

In [18]:
# NLP - generation task
# conditional generation -> generation is depending apon the input

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 406.96it/s]


In [19]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "../results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500,
    # 0 => lr default
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [22]:
# train the model 

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.627839,0.381894
2,0.395926,0.361249
3,0.374032,0.354065
4,0.362793,0.350799
5,0.355544,0.349587
6,0.351616,0.349036


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]


TrainOutput(global_step=3000, training_loss=0.9112916208902995, metrics={'train_runtime': 1825.5123, 'train_samples_per_second': 13.147, 'train_steps_per_second': 1.643, 'total_flos': 3248203235328000.0, 'train_loss': 0.9112916208902995, 'epoch': 6.0})

In [ ]:
# Save the model

model.save_pretrained("../app/saved_summary_model")
tokenizer.save_pretrained("../app/saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("../app/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("../app/saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 870.43it/s]


## Test the core logic for summarization

In [28]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # Clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate summary => token_ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4, # produce 4 summary and return best 1
        early_stopping=True
    )

    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # skip -> EOS, SEP

    return summary

In [29]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
